# Multi-turn Conversation Memory (in-memory)
## Problem Statement
Build an ETL Incident Assistant that maintains conversation context across 8 turns with a sliding window to cap token usage.

## My Approach
Used negative slicing for the sliding window. Append messages every turn,track cumulative tokens via `response.usage.total_tokens`, and once the threshold is exceeded, rebuild `messages` as `[system_msg] + last N messages`. MAX_HISTORY=10 means 5 turns of context are retained after activation.

## What I Learned
LLM memory is entirely manual — the model is stateless, and "memory" is just the messages list you resend each call. The sliding window is a simple but real production pattern: keep cost bounded by discarding old turns while always preserving the system message. Token count grows linearly until the window fires, then stabilizes in a band.

## Where I Got Stuck
in my code , there was `messages[-MAX_HISTORY:]` which can theoretically include the system message in the tail if the list is very short, so the safe pattern is `messages[1:][-MAX_HISTORY:]`.

## What I'd Do Differently
Name MAX_HISTORY → MAX_MESSAGES to make it clear the unit is messages not turns. In a real system, replace hard windowing with a summarization step - instead of dropping old turns, compress them into one "summary so far" message.

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [32]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.1-8b-instant"  
TOKEN_WINDOW_LIMIT = 1500
MAX_HISTORY= 10

SYSTEM_PROMPT = """You are an ETL Incident Assistant for a data engineering team.
You help on-call engineers debug pipeline failures.
Be concise, technical, and retain context across the troubleshooting session."""

messages = [{"role": "system", "content": SYSTEM_PROMPT}]

TEST_CONVERSATION = [
    "Our nightly SSIS package `load_sales_fact` failed at 2:47 AM with error: 'Column name or number of supplied values does not match table definition'.",
    "The source is a flat file from the ERP system. It ran fine yesterday.",
    "We checked the file — it has 47 columns today but the staging table expects 46.",
    "The ERP team says they added a `discount_reason_code` column to the export last night without telling us.",
    "What's the fastest fix to get the pipeline running again without altering the staging table?",
    "OK we used a derived column transform to drop that column. Pipeline ran. Now how do we prevent this from happening again?",
    "What monitoring would you recommend for schema drift on flat file ingestion?",
    "Can you summarize the full incident and the two solutions we discussed?",
]

In [33]:
token_log = []
def chat(user_input):
    global messages

    if token_log and token_log[-1] > TOKEN_WINDOW_LIMIT:
        system_msg = messages[0]
        #My solution(NAIVE)
        # recent_context=messages[-MAX_HISTORY :]
        #Refactored solution
        recent = messages[1:][-MAX_HISTORY:]
        messages = [system_msg] + recent

    messages.append({'role':'user' , 'content' : user_input})

    reponse = client.chat.completions.create(
        model=MODEL
        ,messages=messages
    )

    reply = reponse.choices[0].message.content
    token_count = reponse.usage.total_tokens
    token_log.append(token_count)

    messages.append({"role": "assistant", "content": reply})
    
    return reply , token_count

In [34]:
for turn, user_msg in enumerate(TEST_CONVERSATION, 1):
    print(f"\n--- Turn {turn} ---")
    print(f"User: {user_msg[:80]}...")
    reply, tokens = chat(user_msg)
    print(f"Assistant: {reply[:200]}...")
    print(f"Tokens this turn: {tokens}")


--- Turn 1 ---
User: Our nightly SSIS package `load_sales_fact` failed at 2:47 AM with error: 'Column...
Assistant: Can you please provide more details about the failure, such as:

1. The specific line and error code?
2. The data types of the columns involved in the failed operation?
3. The source and target system...
Tokens this turn: 239

--- Turn 2 ---
User: The source is a flat file from the ERP system. It ran fine yesterday....
Assistant: So the `load_sales_fact` SSIS package was running successfully until yesterday night, and now it's failing with the "Column name or number of supplied values does not match table definition" error.

T...
Tokens this turn: 487

--- Turn 3 ---
User: We checked the file — it has 47 columns today but the staging table expects 46....
Assistant: So, it looks like there's a discrepancy between the number of columns in the source flat file and the number of columns in the staging table. The flat file is expecting 47 columns, but the staging tab...
Token

In [35]:
turns = [i for i in range (1, MAX_HISTORY)]
# Table Header
print(f"{'Turn':<6} | {'Total Tokens':<14} | {'Status'}")
print("-" * 35)

# Table Rows
for turn, count in zip(turns, token_log):
    status = "EXCEEDED" if count > TOKEN_WINDOW_LIMIT else "Under"
    # Highlight Turn 6 as the activation point
    marker = " <--- Window Activated" if count > TOKEN_WINDOW_LIMIT and token_log[turn-2] <= TOKEN_WINDOW_LIMIT else ""
    print(f"{turn:<6} | {count:<14} | {status}{marker}")

Turn   | Total Tokens   | Status
-----------------------------------
1      | 239            | Under
2      | 487            | Under
3      | 752            | Under
4      | 1044           | Under
5      | 1293           | Under
6      | 1683           | EXCEEDED <--- Window Activated
7      | 1994           | EXCEEDED
8      | 2072           | EXCEEDED
